In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
import joblib
import random

# Taille du plateau (doit correspondre à ton interface)
SIZE = 5

In [2]:
def check_winner(board, player):
    q = []
    target_cells = []
    visited = set()
    for i in range(SIZE):
        if player == 1: # IA (Bleu) : Haut -> Bas
            if board[0, i] == 1: q.append((0, i))
            target_cells.append((SIZE-1, i))
        else: # Humain (Rouge) : Gauche -> Droite
            if board[i, 0] == 2: q.append((i, 0))
            target_cells.append((i, SIZE-1))
    while q:
        curr = q.pop(0)
        if curr in target_cells: return True
        if curr in visited: continue
        visited.add(curr)
        r, c = curr
        for dr, dc in [(-1,0), (-1,1), (0,-1), (0,1), (1,-1), (1,0)]:
            nr, nc = r+dr, c+dc
            if 0 <= nr < SIZE and 0 <= nc < SIZE and board[nr, nc] == player:
                q.append((nr, nc))
    return False

In [3]:
def get_smart_move(board, player):
    empty = list(zip(*np.where(board == 0)))

    # 1. Priorité : Gagner immédiatement
    for r, c in empty:
        board[r, c] = player
        if check_winner(board, player):
            board[r, c] = 0
            return (r, c)
        board[r, c] = 0

    # 2. Priorité : Bloquer l'adversaire
    opponent = 3 - player
    for r, c in empty:
        board[r, c] = opponent
        if check_winner(board, opponent):
            board[r, c] = 0
            return (r, c)
        board[r, c] = 0

    # 3. Sinon : Coup aléatoire
    return random.choice(empty)

def encode_board(board):
    encoding = []
    for val in board.flatten():
        if val == 1: encoding.extend([1, 0])
        elif val == 2: encoding.extend([0, 1])
        else: encoding.extend([0, 0])
    return encoding

In [4]:
data_x = []
data_y = []
NUM_GAMES = 5000 # Environ 50 000 à 70 000 positions de jeu

print("Génération en cours...")
for _ in range(NUM_GAMES):
    board = np.zeros((SIZE, SIZE), dtype=int)
    current_player = random.choice([1, 2])

    while True:
        move = get_smart_move(board, current_player)

        # On enregistre la position AVANT le coup si c'est l'IA qui doit jouer
        if current_player == 1:
            data_x.append(encode_board(board))
            data_y.append(move[0] * SIZE + move[1]) # Index du coup (0-24)

        board[move[0], move[1]] = current_player
        if check_winner(board, current_player): break
        if not np.any(board == 0): break
        current_player = 3 - current_player

print(f"Terminé ! {len(data_x)} positions générées.")

Génération en cours...
Terminé ! 46533 positions générées.


In [6]:
print("Entraînement du modèle optimisé...")
# On réduit max_depth et on ajoute min_samples_leaf
model = RandomForestClassifier(
    n_estimators=100,      # Pas besoin de plus pour du 5x5
    max_depth=12,          # Empêche de mémoriser par cœur
    min_samples_leaf=5,    # Force à regrouper des situations similaires
    n_jobs=-1,
    random_state=42
)
model.fit(data_x, data_y)

# Le fichier devrait peser moins de 50 Mo maintenant !
joblib.dump(model, 'hexmodelrfpro_v2.joblib')

Entraînement du modèle optimisé...


['hexmodelrfpro_v2.joblib']